# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

In [8]:
%%configure 
{
    "--conf": "spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.sql.hive.convertMetastoreParquet=false --conf spark.sql.extensions=org.apache.spark.sql.hudi.HoodieSparkSessionExtension",
    "--datalake-formats": "hudi",
    "--enable-metrics": "true",
    "--enable-glue-datacatalog": "true",
    "--enable-continuous-cloudwatch-log": "true",
    "--job-bookmark-option": "job-bookmark-disable",
    "--spark.dynamicAllocation.enabled":"true"
}

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.8 
Previous session type: glueetl
Setting new session type to ETL
Setting session ID prefix to native-hudi-sql-
Setting Glue version to: 3.0
Current idle_timeout is None minutes.
idle_timeout has been set to 60 minutes.
Previous worker type: None
Setting new worker type to: G.2X
Previous number of workers: None
Setting new number of workers to: 20
The following configurations have been updated: {'--conf': 'spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.sql.hive.convertMetastoreParquet=false --conf spark.sql.extensions=org.apache.spark.sql.hudi.HoodieSparkSessionExtension', '--datalake-formats': 'hudi', '--enable-metric

In [5]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType
from pyspark.sql.functions import from_json, col, desc, row_number, when, isnan, isnull, current_timestamp
from pyspark.sql.window import Window
from pyspark import SparkConf
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType, BooleanType, ArrayType, MapType
from pyspark.sql.functions import from_json, col, desc, row_number, when, isnan, isnull, current_timestamp, split, regexp_extract,schema_of_json, lit
import json
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

In [6]:
def infer_data_type(sample_value):
    """
    Infer Spark data type from sample value
    """
    if sample_value is None:
        return StringType()
    
    sample_str = str(sample_value).strip()
    
    # Check for boolean
    if sample_str.lower() in ['true', 'false']:
        return BooleanType()
    
    # Check for integer
    try:
        int(sample_str)
        return IntegerType()
    except ValueError:
        pass
    
    # Check for double/float
    try:
        float(sample_str)
        return DoubleType()
    except ValueError:
        pass
    
    # Check for timestamp patterns
    timestamp_patterns = [
        r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}',  # ISO format
        r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}',  # Standard format
        r'\d{2}/\d{2}/\d{4}',  # Date format
    ]
    
    for pattern in timestamp_patterns:
        if len(sample_str) > 8 and any(char.isdigit() for char in sample_str):
            try:
                # Try to match timestamp pattern
                import re
                if re.search(pattern, sample_str):
                    return TimestampType()
            except:
                pass
    
    # Default to string
    return StringType()

def create_dynamic_schema_from_sample(df_raw, sample_size=1000):
    """
    Create dynamic schema by analyzing sample JSON documents
    """
    print("Creating dynamic schema from sample data...")
    
    # Take sample of raw data
    sample_data = df_raw.limit(sample_size).collect()
    
    schema_fields = {}
    
    for row in sample_data:
        if hasattr(row, '_doc') and row['_doc'] is not None:
            try:
                # Parse JSON document
                doc = json.loads(row['_doc'])
                
                # Analyze each field in the document
                for field_name, field_value in doc.items():
                    if field_name not in schema_fields:
                        # Infer data type for new field
                        data_type = infer_data_type(field_value)
                        schema_fields[field_name] = data_type
                        print(f"Found field: {field_name} -> {data_type}")
                    
            except (json.JSONDecodeError, TypeError) as e:
                print(f"Error parsing JSON document: {e}")
                continue
    
    # Build StructType schema
    struct_fields = []
    for field_name, data_type in schema_fields.items():
        struct_fields.append(StructField(field_name, data_type, True))
    
    dynamic_schema = StructType(struct_fields)
    
    print(f"Dynamic schema created with {len(struct_fields)} fields:")
    for field in struct_fields:
        print(f"  - {field.name}: {field.dataType}")
    
    return dynamic_schema

# Main processing code
source_path = "s3://fygluetest/sample_data/data_3/"

print("Step 1: Reading raw data from source...")
df_raw = spark.read.format("parquet").load(source_path)

print(f"Raw record count: {df_raw.count()}")
print("Raw data schema:")
df_raw.printSchema()

# Method 1: Dynamic schema from sample data
print("\n=== Method 1: Creating dynamic schema from sample data ===")
try:
    sample_json = df_raw.select("_doc").filter(col("_doc").isNotNull()).limit(1).collect()
    if sample_json:
        doc_sample_str = sample_json[0]['_doc']
        schema_to_use= schema_of_json(lit(doc_sample_str))
        print("Schema Formation Completely")
except Exception as e:
    print(f"Error creating dynamic schema: {e}")
    print("Falling back to flexible schema")
    dynamic_schema = create_dynamic_schema_from_sample(df_raw, sample_size=100)
    schema_to_use = dynamic_schema
    print("Using dynamically inferred schema")
    # schema_to_use = create_flexible_schema()


Step 1: Reading raw data from source...
Raw record count: 11
Raw data schema:
root
 |-- Op: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- _doc: string (nullable = true)


=== Method 1: Creating dynamic schema from sample data ===


In [7]:
print("Step 2: Parsing JSON documents from _doc field...")
# Parse the _doc JSON field and extract all nested fields

df_parsed = (df_raw
             .withColumn("parsed_doc", from_json(col("_doc"), schema_to_use))
             .select("*", "parsed_doc.*")  # This keeps all original columns AND adds parsed fields
             .withColumn("operation", 
                        col("Op") if "Op" in df_raw.columns else col("op"))
             .drop("parsed_doc"))  # Remove intermediate column

print(f"Parsed record count: {df_parsed.count()}")
print("Parsed data schema:")
df_parsed.printSchema()

# Show sample parsed data
print("Sample parsed data (first 5 rows):")
df_parsed.show(15, truncate=False)


Step 2: Parsing JSON documents from _doc field...
Parsed record count: 11
Parsed data schema:
root
 |-- Op: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- _doc: string (nullable = true)
 |-- _id: struct (nullable = true)
 |    |-- $oid: string (nullable = true)
 |-- date: string (nullable = true)
 |-- name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- operation: string (nullable = true)

Sample parsed data (first 5 rows):
+----+--------------------------+---------------------------------------------------------------------------------------------------------------------------+--------------------------+----------+-----+----------------+---------+
|Op  |timestamp                 |_doc                                                                                                                       |_id                       |date      |name |status          |operation|
+----+--------------------------+------------------------------------

In [8]:
print("Step 3: Filtering out invalid records...")
df_valid = df_parsed.filter(col("_id").isNotNull())
print(f"Valid records count: {df_valid.count()}")

Step 3: Filtering out invalid records...
Valid records count: 11


In [9]:
print("Step 4: Performing deduplication...")
# Create window specification for deduplication
# Partition by _id and order by last_modified descending to get latest record
windowSpec = Window.partitionBy("_id").orderBy(desc("timestamp"))

# Add row number and keep only the first row (latest) for each _id
df_with_rownum = df_valid.withColumn("row_number", row_number().over(windowSpec))

# Filter to keep only the latest record for each _id
df_deduped = df_with_rownum.filter(col("row_number") == 1).drop("row_number")

print(f"Deduplicated record count: {df_deduped.count()}")

Step 4: Performing deduplication...
Deduplicated record count: 9


In [10]:
print("Step 5: Handling CDC operations and deleted records...")
# Mark deleted records based on CDC operation or deleted flag
df_final = df_deduped.withColumn("is_deleted", 
    when((col("Op") == "D"), True)
    .otherwise(False))


df_final.show(15, truncate=False)


df_final = df_final.filter( (col("Op").isNull()) | (col("Op") == "") | (col("Op") != "D"))

# Optional: Filter out deleted records if you don't want to keep them
# Uncomment the line below if you want to exclude deleted records
# df_final = df_final.filter(col("is_deleted") == False)

# Add processing metadata
df_final = df_final.withColumn("processed_at", current_timestamp())

print(f"Final processed record count: {df_final.count()}")

# Show final data sample
print("Final processed data sample:")
df_final.select("_id", "name", "status", "timestamp", "is_deleted", "processed_at").show(10)

# Show summary statistics
print("Summary by status:")
df_final.groupBy("status").count().show()

print("Summary by is_deleted flag:")
df_final.groupBy("is_deleted").count().show()

# Optional: Check for any remaining duplicates (should be 0)
duplicate_check = df_final.groupBy("_id").count().filter(col("count") > 1)
duplicate_count = duplicate_check.count()
print(f"Remaining duplicates: {duplicate_count}")

if duplicate_count > 0:
    print("Warning: Still have duplicates after deduplication!")
    duplicate_check.show()

print("Deduplication process completed successfully!")

Step 5: Handling CDC operations and deleted records...
+----+--------------------------+---------------------------------------------------------------------------------------------------------------------------+--------------------------+----------+-----+----------------+---------+----------+
|Op  |timestamp                 |_doc                                                                                                                       |_id                       |date      |name |status          |operation|is_deleted|
+----+--------------------------+---------------------------------------------------------------------------------------------------------------------------+--------------------------+----------+-----+----------------+---------+----------+
|U   |2025-06-03 04:56:23.000001|{ "_id" : { "$oid" : "682c2637dda06d2d7593446e" }, "name" : "test1", "date" : "21-05-2025", "status" : "updated_03/06/25" }|{682c2637dda06d2d7593446e}|21-05-2025|test1|updated_03/06/25|U      

In [27]:
# Define Hudi target path
target_path = "s3://fygluetest/mongodb-hudi/collection-name/"

# Enhanced Hudi options with proper configuration
hudi_options = {
    'hoodie.table.name': 'mongodb_hudi_table',
    'hoodie.datasource.write.recordkey.field': '_id',
    'hoodie.datasource.write.precombine.field': 'timestamp',
    'hoodie.datasource.write.operation': 'upsert',
    'hoodie.datasource.write.table.type': 'COPY_ON_WRITE',
    'hoodie.upsert.shuffle.parallelism': 10,
    'hoodie.insert.shuffle.parallelism': 10,
    'hoodie.bulkinsert.shuffle.parallelism': 10,
    'hoodie.datasource.hive_sync.enable': 'true',
    'hoodie.datasource.hive_sync.database': 'my_glue_db',
    'hoodie.datasource.hive_sync.table': 'mongodb_hudi_table',
    'hoodie.datasource.hive_sync.use_jdbc': 'false',
    'hoodie.datasource.hive_sync.mode': 'hms',
    'hoodie.datasource.hive_sync.support_timestamp': 'true',
    'hoodie.datasource.hive_sync.partition_extractor_class': 'org.apache.hudi.hive.MultiPartKeysValueExtractor',
    'hoodie.datasource.hive_sync.use_jdbc': 'false',
    'hoodie.datasource.hive_sync.mode': 'hms',
    'hoodie.datasource.hive_sync.support_timestamp': 'true'
}


print("Writing to Hudi table...")
try:
    df_final.write.format("hudi") \
        .options(**hudi_options) \
        .mode("append") \
        .save(target_path)
    print("Successfully written to Hudi table!")
except Exception as e:
    print(f"Error writing to Hudi: {str(e)}")
    raise e

Writing to Hudi table...
Successfully written to Hudi table!


####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [ ]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='database_name', table_name='table_name')
dyf.printSchema()

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [ ]:
df = dyf.toDF()
df.show()

#### Example: Visualize data with matplotlib


In [ ]:
import matplotlib.pyplot as plt

# Set X-axis and Y-axis values
x = [5, 2, 8, 4, 9]
y = [10, 4, 8, 5, 2]
  
# Create a bar chart 
plt.bar(x, y)
  
# Show the plot
%matplot plt

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [ ]:
s3output = glueContext.getSink(
  path="s3://bucket_name/folder_name",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="demo", catalogTableName="populations"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(DyF)